In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot
import statsmodels.api as sm
from ISLP import load_data
from ISLP.models import (ModelSpec as MS, summarize)
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [2]:
Weekly = load_data("Weekly")
Weekly

,Year,Lag1,Lag2,Lag3,Lag4,Lag5,Volume,Today,Direction
0,1990,0.816,1.572,-3.936,-0.229,-3.484,0.154976,-0.270,Down
1,1990,-0.270,0.816,1.572,-3.936,-0.229,0.148574,-2.576,Down
2,1990,-2.576,-0.270,0.816,1.572,-3.936,0.159837,3.514,Up
3,1990,3.514,-2.576,-0.270,0.816,1.572,0.161630,0.712,Up
4,1990,0.712,3.514,-2.576,-0.270,0.816,0.153728,1.178,Up
...,...,...,...,...,...,...,...,...,...
1084,2010,-0.861,0.043,-2.173,3.599,0.015,3.205160,2.969,Up
1085,2010,2.969,-0.861,0.043,-2.173,3.599,4.242568,1.281,Up
1086,2010,1.281,2.969,-0.861,0.043,-2.173,4.835082,0.283,Up
1087,2010,0.283,1.281,2.969,-0.861,0.043,4.454044,1.034,Up


In [5]:
allvars = Weekly.columns.drop(['Today', 'Direction', 'Year', 'Lag3', 'Lag4', 'Lag5', 'Volume'])
design = MS(allvars)
X = design.fit_transform(Weekly)
y = Weekly.Direction == 'Up'
glm = sm.GLM(y, X, family=sm.families.Binomial())
results = glm.fit()
summarize(results)

,coef,std err,z,P>|z|
intercept,0.2212,0.061,3.599,0.000
Lag1,-0.0387,0.026,-1.477,0.140
Lag2,0.0602,0.027,2.270,0.023


In [9]:
allvars_reduced = Weekly.loc[1:, allvars]
design = MS(allvars_reduced)
X = design.fit_transform(Weekly.loc[1:])
y = Weekly.loc[1:, 'Direction'] == 'Up'
glm = sm.GLM(y, X, family=sm.families.Binomial())
results = glm.fit()
summarize(results)

,coef,std err,z,P>|z|
intercept,0.2232,0.061,3.630,0.000
Lag1,-0.0384,0.026,-1.466,0.143
Lag2,0.0608,0.027,2.291,0.022


In [17]:
first_observation = X.iloc[[0]]
predicted_prob = results.predict(first_observation)

In [18]:
predicted_direction = 'Up' if predicted_prob.iloc[0] > 0.5 else 'Down'

In [19]:
print(predicted_direction)

Up


In [20]:
actual_direction = 'Up' if Weekly.loc[0, 'Direction'] == 'Up' else 'Down'

In [21]:
actual_direction

'Down'

In [38]:
n = Weekly.shape[0]
actual = Weekly['Direction'] == 'Up'
errors = 0

In [39]:
preds = Weekly[allvars]    
design = MS(preds)      


for i in range(n):
    Weekly_red = Weekly.loc[Weekly.index != i]
    train_X = design.fit_transform(Weekly_red[allvars])
    train_y = Weekly_red.Direction == 'Up'

    glm = sm.GLM(train_y, train_X, family=sm.families.Binomial())
    results = glm.fit()

    test_X = design.transform(Weekly.loc[[Weekly.index[i]], allvars])

    predicted_prob = results.predict(test_X)
    predicted_direction = 'Up' if predicted_prob.iloc[0] > 0.5 else 'Down'
    actual_i = actual.iloc[i]

    error = int((predicted_direction == 'Up') != actual_i)
    errors += error


In [40]:
errors

490

In [41]:
errors / n

0.44995408631772266